<a href="https://colab.research.google.com/github/Nick738996/Algoritmos-de-Optimizacion/blob/main/Seminario_Algoritmos_Brandon_Gomez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Seminario<br>
Nombre y Apellidos: Brandon Nick Gomez Aya  <br>
Url: https://github.com/Nick738996/Algoritmos-de-Optimizacion/blob/main/Seminario_Algoritmos_Brandon_Gomez.ipynb<br>
Problema:
> 1. Sesiones de doblaje <br>
>2. Organizar los horarios de partidos de La Liga<br>
>3. Combinar cifras y operaciones

Descripción del problema:(copiar enunciado)

Problema 1. Organizar sesiones de doblaje(I)
• Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en las
tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de
doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el estudio de
grabación independientemente del número de tomas que se graben. No es posible grabar más
de 6 tomas por día. El objetivo es planificar las sesiones por día de manera que el gasto por los
servicios de los actores de doblaje sea el menor posible. Los datos son:
Número de actores: 10
Número de tomas : 30
Actores/Tomas : https://bit.ly/36D8IuK
- 1 indica que el actor participa en la toma
- 0 en caso contrario

(*) La respuesta es obligatoria                               

(*)¿Cuantas posibilidades hay sin tener en cuenta las restricciones?<br>



¿Cuantas posibilidades hay teniendo en cuenta todas las restricciones.




Respuesta

Para no quedarnos solo en la teoría, calculé esto usando combinatoria básica.

**1. Sin tener en cuenta las restricciones:**
Si somos flexibles y asumimos el mínimo de días necesarios (30 tomas / 6 por día = 5 días base), cada una de las 30 tomas podría caer en cualquiera de esos 5 días. Esto es una variación con repetición simple: $5^{30}$. Es un número absurdamente gigante. (Y eso sin contar si decidimos grabar en más de 5 días, lo que elevaría las opciones muchísimo más).

**2. Teniendo en cuenta las restricciones:**
Aquí la cosa se pone estricta. Necesitamos repartir exactamente 30 tomas en 5 días, y cada día debe tener exactamente 6 tomas. Matemáticamente, esto es agrupar 30 elementos en 5 subconjuntos iguales de 6 elementos.
La fórmula combinatoria es: $\frac{30!}{(6!)^5 \times 5!}$ (dividimos por 5! para no contar como válidas las permutaciones de los mismos días, ya que el orden de los días no afecta el costo final).

A continuación, dejo el cálculo en código para ver las cifras exactas:

In [48]:
import math

# 1. Cálculo SIN restricciones (Asumiendo 5 días de grabación)
# Cada una de las 30 tomas tiene 5 opciones posibles de día
sin_restricciones = 5**30

print("ESPACIO DE SOLUCIONES")
print(f"1. Posibilidades SIN restricciones (aprox): {sin_restricciones:,}")


# 2. Cálculo CON restricciones (5 días exactos, 6 tomas exactas por día)
# Aplicamos la fórmula: 30! / ( (6!)^5 * 5! )
numerador = math.factorial(30)
denominador = (math.factorial(6)**5) * math.factorial(5)

con_restricciones = numerador // denominador

print(f"2. Posibilidades CON restricciones:       {con_restricciones:,}")

ESPACIO DE SOLUCIONES
1. Posibilidades SIN restricciones (aprox): 931,322,574,615,478,515,625
2. Posibilidades CON restricciones:       11,423,951,396,577,720


Modelo para el espacio de soluciones<br>
(*) ¿Cual es la estructura de datos que mejor se adapta al problema? Argumentalo.(Es posible que hayas elegido una al principio y veas la necesidad de cambiar, arguentalo)


Respuesta

Para este problema, pasé por un proceso de evolución en cuanto a la estructura de datos elegida. Lo dividí en dos partes: cómo almacenar la información base y cómo representar la solución.

**1. Para los datos de entrada (La matriz de tomas y actores):**
Al principio pensé en usar un diccionario donde cada clave fuera la toma y el valor fuera una lista de actores. Sin embargo, vi la necesidad de cambiarlo a una **Matriz bidimensional (lista de listas en Python o array de NumPy)** de 30x10.
* *Argumento:* Trabajar con una matriz binaria (1 si participa, 0 si no) nos permite aprovechar la indexación. Buscar si un actor participa en una toma específica se vuelve una operación de tiempo constante $O(1)$ haciendo algo como `matriz[toma][actor]`.

**2. Para el espacio de soluciones (La planificación de días):**
Inicialmente pensé en crear un arreglo unidimensional de tamaño 30, donde el índice fuera la toma y el valor fuera el día asignado (ejemplo: `solucion[0] = 1` significaría que la toma 1 se graba el día 1).
Pero me di cuenta de que esto era un error. Validar la restricción de "máximo 6 tomas por día" iba a ser muy ineficiente porque tocaría recorrer y contar todo el arreglo constantemente.

* *La estructura definitiva:* Decidí cambiarlo y la estructura que mejor se adapta es una **Lista de Listas** (donde cada sublista representa un día de grabación).
Ejemplo: `Planificacion = [ [Toma1, Toma5...], [Toma2, Toma8...], ... ]`

* *Argumento del cambio:* Esta estructura me soluciona la vida de dos formas:
  a) Validar restricciones es trivial: solo tengo que hacer `len(Planificacion[dia])`. Si el tamaño es 6, sé que ese día está lleno y no puedo agregar más tomas.
  b) Calcular la función objetivo es directísimo: para saber cuántos actores van en un día, recorro las tomas de esa sublista, extraigo los actores que tienen un '1' y los inserto en una estructura de conjunto (`set()` en Python) que automáticamente me elimina los actores duplicados.

In [49]:
import pandas as pd
from google.colab import drive

# 1. Conectamos con tu Google Drive personal
drive.mount('/content/drive')

# 2. Cargamos el archivo CSV desde el Drive.
# OJO: Por defecto, si el archivo está suelto en tu Drive, la ruta es esta.
# Si lo metiste dentro de alguna carpeta, tendrías que sumarle el nombre de la carpeta.
ruta_archivo = '/content/drive/MyDrive/Copy of Datos problema doblaje(30 tomas, 10 actores) - Hoja 1.csv'

# Ahora sí leemos el archivo y creamos el famoso 'df'
df = pd.read_csv(ruta_archivo, header=1)

# 3. Filtramos solo las columnas de los 10 actores (ignoramos 'Toma' y 'Total')
columnas_actores = [str(i) for i in range(1, 11)]

# 4. Convertimos esos datos a nuestra estructura elegida: Una matriz (Lista de listas)
matriz_tomas = df[columnas_actores].fillna(0).astype(int).values.tolist()

print("--- ESTRUCTURA DE DATOS CARGADA ---")
print(f"Total de tomas cargadas: {len(matriz_tomas)}")
print(f"Total de actores por toma: {len(matriz_tomas[0])}")
print("\nEjemplo de cómo se ve la Toma 1 internamente (1=Participa, 0=No participa):")
print(matriz_tomas[0])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- ESTRUCTURA DE DATOS CARGADA ---
Total de tomas cargadas: 32
Total de actores por toma: 10

Ejemplo de cómo se ve la Toma 1 internamente (1=Participa, 0=No participa):
[1, 1, 1, 1, 1, 0, 0, 0, 0, 0]


Según el modelo para el espacio de soluciones<br>
(*)¿Cual es la función objetivo?

(*)¿Es un problema de maximización o minimización?

Respuesta

Definitivamente es un problema de **minimización**. Lo que buscamos es reducir al máximo el dinero que se gasta la productora pagándole a los actores por cada día que van al estudio.

**Función Objetivo:**
Como a los actores se les paga por el día que asisten (sin importar si graban 1 toma o el máximo de 6 tomas), el costo de un día de grabación depende exclusivamente de cuántos actores *distintos* (únicos) participan en la jornada.

Entonces, la función objetivo es minimizar la suma de actores únicos requeridos por día, sumando todos los días que dure la grabación.

En lenguaje matemático se vería algo así:
$$Minimizar \ Z = \sum_{j=1}^{D} |A_j|$$

Donde:
* $D$ es el número total de días de grabación.
* $A_j$ es el conjunto de actores *únicos* que asisten el día $j$. (Al tratarlo como un conjunto, nos aseguramos de no contar a un mismo actor dos veces si participa en varias tomas el mismo día).

In [50]:
def costo_dia(tomas_dia, matriz_tomas):
    """
    Calcula el costo de un día de grabación (número de actores únicos).
    tomas_dia: Lista con los índices de las tomas asignadas a ese día (ej: [0, 5, 12])
    matriz_tomas: Nuestra estructura de datos principal
    """
    actores_necesarios = set() # Usamos un 'set' (conjunto) para evitar contar actores repetidos

    for toma in tomas_dia:
        # Recorremos la matriz en la fila de la toma específica
        for actor_idx, participa in enumerate(matriz_tomas[toma]):
            if participa == 1:
                # Si el actor participa, lo agregamos al conjunto
                actores_necesarios.add(actor_idx)

    # El costo es simplemente cuántos actores diferentes quedaron en el conjunto
    return len(actores_necesarios)

def costo_total_planificacion(planificacion, matriz_tomas):
    """
    Suma el costo de todos los días de la planificación (Función Z completa).
    planificacion: Lista de listas (ej: [[0,1,2,3,4,5], [6,7,8,9,10,11], ...])
    """
    costo_total = 0
    for tomas in planificacion:
        costo_total += costo_dia(tomas, matriz_tomas)

    return costo_total

# PRUEBA DE LA FUNCIÓN OBJETIVO
# Vamos a simular que el Día 1 grabamos las primeras 6 tomas (índices del 0 al 5)
dia_de_prueba = [0, 1, 2, 3, 4, 5]
costo = costo_dia(dia_de_prueba, matriz_tomas)

print("PRUEBA DE FUNCIÓN OBJETIVO")
print(f"Tomas asignadas para hoy: {dia_de_prueba}")
print(f"Total de actores distintos a pagar hoy: {costo}")

PRUEBA DE FUNCIÓN OBJETIVO
Tomas asignadas para hoy: [0, 1, 2, 3, 4, 5]
Total de actores distintos a pagar hoy: 7


Diseña un algoritmo para resolver el problema por fuerza bruta

Respuesta

A nivel teórico, el diseño para resolver este problema por fuerza bruta se basa en un algoritmo recursivo de **Vuelta Atrás (Backtracking)**.

El algoritmo explora el árbol completo de decisiones con la siguiente lógica:
1. Tomamos la primera toma de la lista.
2. Intentamos asignarla al "Día 1". Si el Día 1 ya tiene 6 tomas (restricción máxima), intentamos en el Día 2, y así sucesivamente.
3. Si la toma no cabe en ningún día existente, abrimos un nuevo día de grabación.
4. Hacemos una llamada recursiva para acomodar la siguiente toma.
5. Cuando llegamos a una hoja del árbol (todas las tomas asignadas), calculamos el costo total de esa planificación con nuestra función objetivo. Si es el costo más bajo que hemos visto, lo guardamos como nuestro "récord".
6. Aplicamos el *Backtracking*: sacamos la última toma que acomodamos y la intentamos poner en otro día diferente para explorar otra rama de combinaciones.

*Nota:* Debido a la explosión combinatoria ($O(D^N)$), ejecutar este algoritmo para las 30 tomas es computacionalmente inviable. Por ello, en la celda de código adjunta, presento la implementación pero la pruebo únicamente con un subconjunto de 6 tomas para demostrar su correcto funcionamiento.

In [51]:
# ALGORITMO DE FUERZA BRUTA (Backtracking)

def fuerza_bruta_doblaje(tomas_restantes, planificacion_actual, mejor_costo, mejor_plan):
    # Condición de parada: Si ya acomodamos todas las tomas de la lista
    if not tomas_restantes:
        # Evaluamos cuánto cuesta esta configuración exacta
        costo_actual = costo_total_planificacion(planificacion_actual, matriz_tomas)

        # Si es mejor que nuestro récord, actualizamos
        if costo_actual < mejor_costo:
            # Hacemos una copia profunda de la planificación para no perderla por referencia
            return costo_actual, [dia[:] for dia in planificacion_actual]
        return mejor_costo, mejor_plan

    toma_actual = tomas_restantes[0]

    # 1. Intentamos meter la toma en algún día existente que tenga cupo (< 6 tomas)
    for dia in planificacion_actual:
        if len(dia) < 6:
            dia.append(toma_actual)
            # Llamada recursiva con el resto de las tomas
            mejor_costo, mejor_plan = fuerza_bruta_doblaje(tomas_restantes[1:], planificacion_actual, mejor_costo, mejor_plan)
            # ¡Vuelta atrás (Backtrack)! Sacamos la toma para probar otra ruta
            dia.pop()

    # 2. También intentamos abrir un día completamente nuevo para esta toma
    planificacion_actual.append([toma_actual])
    mejor_costo, mejor_plan = fuerza_bruta_doblaje(tomas_restantes[1:], planificacion_actual, mejor_costo, mejor_plan)
    # ¡Vuelta atrás! Cerramos el día que abrimos
    planificacion_actual.pop()

    return mejor_costo, mejor_plan

tomas_prueba = [0, 1, 2, 3, 4, 5] # Índices de las primeras 6 tomas

# Arrancamos con infinito y una planificación vacía
costo_fb, plan_fb = fuerza_bruta_doblaje(tomas_prueba, [], float('inf'), [])

print("RESULTADO FUERZA BRUTA (MUESTRA PEQUEÑA)")
print(f"Mejor costo encontrado para las 6 tomas: {costo_fb} actores")
print(f"Planificación ideal (Agrupación de tomas): {plan_fb}")

RESULTADO FUERZA BRUTA (MUESTRA PEQUEÑA)
Mejor costo encontrado para las 6 tomas: 7 actores
Planificación ideal (Agrupación de tomas): [[0, 1, 2, 3, 4, 5]]


Calcula la complejidad del algoritmo por fuerza bruta

Respuesta

La complejidad de nuestro algoritmo de fuerza bruta (Backtracking) es de orden **Exponencial**.

Matemáticamente se expresa como **$O(D^N)$** en el peor de los casos.

**Justificación:**
* $N$ es el número total de tomas a planificar (profundidad del árbol recursivo).
* $D$ es el número de días o "grupos" disponibles en los que podemos asignar una toma (el factor de ramificación).

En cada paso de la recursividad, el algoritmo toma 1 escena y debe decidir en cuál de los $D$ días meterla. Como esto se repite para las $N$ tomas, el número de nodos a explorar en el árbol crece exponencialmente.

Si asumimos un límite estricto de 5 días de grabación para las 30 tomas, la complejidad sería de **$O(5^{30})$** operaciones.

A esto hay que sumarle el costo de la validación de las restricciones (comprobar el límite de 6 tomas y sumar los actores con conjuntos), que nos tomaría $O(A)$ donde $A$ es el número de actores. Por lo tanto, la complejidad temporal total rondaría los **$O(A \cdot D^N)$**.

*Conclusión:* Es una complejidad inmanejable para $N=30$. Nos confirma que para el problema real debemos descartar la fuerza bruta y pasar a diseñar un algoritmo Heurístico o Voraz (Greedy) que nos dé una solución subóptima pero en un tiempo razonable, con complejidad polinómica.

In [52]:
# DEMOSTRACIÓN DEL CRECIMIENTO EXPONENCIAL O(5^N)

def estimar_tiempo_fuerza_bruta(num_tomas, dias_disponibles=5):
    # Asumimos que un PC moderno hace 1 billón de operaciones por segundo
    operaciones_por_segundo = 1_000_000_000

    # Operaciones estimadas en el peor caso O(D^N)
    operaciones_totales = dias_disponibles ** num_tomas

    # Cálculo de tiempo
    segundos = operaciones_totales / operaciones_por_segundo
    minutos = segundos / 60
    horas = minutos / 60
    dias = horas / 24
    anios = dias / 365

    return operaciones_totales, anios

print("TIEMPO ESTIMADO PARA FUERZA BRUTA")

# Vamos a ver cómo crece el tiempo a medida que añadimos tomas
for tomas in [5, 10, 15, 30]:
    ops, tiempo_anios = estimar_tiempo_fuerza_bruta(tomas)
    if tomas == 30:
        print(f"\nPara el problema real ({tomas} tomas):")
        print(f"- Operaciones estimadas: {ops:,}")
        print(f"- Tiempo estimado: ¡{tiempo_anios:,.0f} años!")
    else:
        print(f"Para {tomas} tomas tarda una fracción de segundo.")

TIEMPO ESTIMADO PARA FUERZA BRUTA
Para 5 tomas tarda una fracción de segundo.
Para 10 tomas tarda una fracción de segundo.
Para 15 tomas tarda una fracción de segundo.

Para el problema real (30 tomas):
- Operaciones estimadas: 931,322,574,615,478,515,625
- Tiempo estimado: ¡29,532 años!


(*)Diseña un algoritmo que mejore la complejidad del algortimo por fuerza bruta. Argumenta porque crees que mejora el algoritmo por fuerza bruta

Respuesta

Para mejorar drásticamente la complejidad, he diseñado un **Algoritmo Voraz (Greedy)**.

**¿Por qué mejora al algoritmo por fuerza bruta?**
El algoritmo de fuerza bruta intenta explorar *todas* las combinaciones posibles (árbol de decisiones completo), lo que nos da un tiempo de ejecución exponencial $O(D^N)$ que tardaría millones de años.

En cambio, el algoritmo voraz no mira hacia el futuro ni explora todas las ramas. En cada paso, toma la decisión que parece ser la mejor en ese instante (óptimo local).

**La lógica (heurística) del algoritmo es la siguiente:**
1. **Ordenamiento:** Primero, ordenamos todas las tomas de mayor a menor según la cantidad de actores que requieren. (Es mejor acomodar primero las escenas más pesadas y dejar las más fáciles para el final).
2. **Asignación:** Iteramos sobre cada toma y buscamos en qué día de los ya creados genera el **menor impacto económico** (es decir, en qué día añade la menor cantidad de actores *nuevos*).
3. **Restricciones:** Validamos que el día elegido no tenga ya 6 tomas. Si ningún día tiene espacio, simplemente abrimos un nuevo día de grabación.

**Complejidad lograda:**
Pasamos de un tiempo $O(D^N)$ a un tiempo polinómico $O(N \log N + N \cdot D)$. El $N \log N$ viene de ordenar las tomas al principio, y el $N \cdot D$ de comparar cada toma con los días existentes. ¡Esto se ejecuta en milisegundos!

In [53]:
# ALGORITMO VORAZ (GREEDY) PARA SESIONES DE DOBLAJE

def algoritmo_voraz_doblaje(matriz):
    # 1. Preprocesamiento: Crear una lista de tuplas (indice_toma, conjunto_de_actores)
    tomas_info = []
    for i, fila in enumerate(matriz):
        # Guardamos solo los actores que tienen un 1 (los que participan)
        actores = {j for j, participa in enumerate(fila) if participa == 1}
        tomas_info.append((i, actores))

    # 2. Heurística: Ordenar las tomas por número de actores (de mayor a menor)
    tomas_info.sort(key=lambda x: len(x[1]), reverse=True)

    # Lista para guardar los días de grabación.
    # Cada día será un diccionario con las 'tomas' asignadas y los 'actores' únicos de ese día
    dias_grabacion = []

    # 3. Asignación Voraz
    for idx_toma, actores_toma in tomas_info:
        mejor_dia = -1
        menor_incremento = float('inf')

        # Revisamos los días que ya hemos abierto para ver dónde sale más barato meter la toma
        for i, dia in enumerate(dias_grabacion):
            if len(dia['tomas']) < 6: # Restricción estricta: Máximo 6 tomas por día

                # Calculamos cuántos actores NUEVOS habría que pagar si metemos la toma hoy
                actores_nuevos = len(actores_toma - dia['actores'])

                if actores_nuevos < menor_incremento:
                    menor_incremento = actores_nuevos
                    mejor_dia = i

        # Decidimos: ¿Lo metemos en el mejor día existente o abrimos uno nuevo?
        if mejor_dia != -1:
            # Lo asignamos al día donde menos actores nuevos añade
            dias_grabacion[mejor_dia]['tomas'].append(idx_toma)
            dias_grabacion[mejor_dia]['actores'].update(actores_toma)
        else:
            # No hay espacio en ningún día, abrimos uno nuevo
            dias_grabacion.append({'tomas': [idx_toma], 'actores': set(actores_toma)})

    # 4. Formatear la salida para que sea compatible con nuestra función de costo
    planificacion_final = [dia['tomas'] for dia in dias_grabacion]
    return planificacion_final

# EJECUCIÓN CON LOS DATOS REALES (LAS 30 TOMAS)

plan_voraz = algoritmo_voraz_doblaje(matriz_tomas)
costo_voraz = costo_total_planificacion(plan_voraz, matriz_tomas)

print("RESULTADO ALGORITMO VORAZ")
print(f"Total de días planificados: {len(plan_voraz)}")
print(f"Costo total (Actores a pagar): {costo_voraz}")
print("\nPlanificación detallada (Índices de las tomas por día):")
for i, dia in enumerate(plan_voraz):
    print(f"Día {i+1}: Tomas {dia}")

RESULTADO ALGORITMO VORAZ
Total de días planificados: 6
Costo total (Actores a pagar): 38

Planificación detallada (Índices de las tomas por día):
Día 1: Tomas [0, 10, 11, 3, 5, 6]
Día 2: Tomas [9, 19, 21, 24, 25, 1]
Día 3: Tomas [2, 4, 7, 8, 12, 13]
Día 4: Tomas [14, 28, 15, 16, 17, 18]
Día 5: Tomas [20, 22, 23, 26, 27, 29]
Día 6: Tomas [30, 31]


(*)Calcula la complejidad del algoritmo

Respuesta

La complejidad temporal de nuestro algoritmo Voraz (Greedy) es **polinómica**, específicamente del orden de **$O(N \log N + N \cdot D \cdot A)$**.

**Desglose del cálculo paso a paso:**
1. **Preprocesamiento:** Leer la matriz para convertirla en conjuntos de actores toma $O(N \cdot A)$, donde $N$ es el número de tomas y $A$ el número de actores.
2. **Ordenamiento (Heurística):** Ordenar las $N$ tomas de mayor a menor cantidad de actores usando el algoritmo nativo de Python (Timsort) tiene una complejidad de **$O(N \log N)$**.
3. **Bucle de Asignación:** Iteramos sobre las $N$ tomas. Por cada toma, revisamos los $D$ días de grabación que hemos abierto. Dentro de esa revisión, hacemos una resta de conjuntos de actores (que toma un tiempo proporcional a $A$). Esto nos da una complejidad de **$O(N \cdot D \cdot A)$**.

**Conclusión:**
Como el número de actores ($A=10$) es una constante pequeña, y el número de días ($D$) crece muy lentamente (y siempre será menor a $N$), el algoritmo se comporta en la práctica como $O(N \log N + N^2)$.

¡Esta es una mejora monumental! Pasamos de una complejidad exponencial $O(D^N)$ que tardaba millones de años, a una complejidad polinómica que encuentra una solución altamente optimizada en un par de milisegundos.

In [54]:
import time

# DEMOSTRACIÓN DE COMPLEJIDAD (TIEMPO DE EJECUCIÓN REAL)

print("MIDIENDO EL TIEMPO DEL ALGORITMO VORAZ")

# Tomamos el tiempo exacto antes de empezar
inicio = time.time()

# Ejecutamos nuestro algoritmo con las 30 tomas reales
plan_voraz_prueba = algoritmo_voraz_doblaje(matriz_tomas)

# Tomamos el tiempo al terminar
fin = time.time()

# Calculamos la diferencia en milisegundos
tiempo_ejecucion = (fin - inicio) * 1000

print(f"El algoritmo resolvió las {len(matriz_tomas)} tomas en: {tiempo_ejecucion:.4f} milisegundos.")
print("¡Confirmado! La complejidad es polinómica y súper eficiente comparada con los 30 millones de años de la Fuerza Bruta.")

MIDIENDO EL TIEMPO DEL ALGORITMO VORAZ
El algoritmo resolvió las 32 tomas en: 0.1206 milisegundos.
¡Confirmado! La complejidad es polinómica y súper eficiente comparada con los 30 millones de años de la Fuerza Bruta.


Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios

Respuesta

Tiene todo el sentido del mundo! Para comprobar la robustez de nuestro algoritmo Voraz, vamos a crear un "Generador de Guiones Aleatorios".

**Diseño del juego de datos:**
He diseñado una función que recibe la cantidad de tomas (filas) y la cantidad de actores (columnas) que queramos inventarnos.
Utilizando la librería `random`, el código decide si un actor participa o no en una toma basándose en una probabilidad configurable (por ejemplo, un 25% de probabilidad de aparecer).

Además, le agregué una validación de seguridad: si por pura casualidad el azar genera una fila llena de ceros (una toma sin actores), el código fuerza la entrada de al menos un actor aleatorio, garantizando que todas las tomas del guion sean válidas y tengan sentido.

El resultado es una matriz binaria con la estructura exacta que requiere nuestro algoritmo.

In [55]:
import random

# GENERADOR DE DATOS ALEATORIOS (NUEVA PELÍCULA)


def generar_guion_aleatorio(num_tomas, num_actores, prob_aparicion=0.3):
    """
    Genera una matriz binaria simulando las tomas de una nueva película.
    prob_aparicion: Probabilidad (de 0 a 1) de que un actor participe en una toma.
    """
    matriz_aleatoria = []

    for _ in range(num_tomas):
        toma = []
        for _ in range(num_actores):
            # Genera un 1 con la probabilidad dada, si no, un 0
            if random.random() < prob_aparicion:
                toma.append(1)
            else:
                toma.append(0)

        # Validación: Si por azar una toma se generó sin ningún actor,
        # metemos a un actor al azar para que la toma tenga sentido.
        if sum(toma) == 0:
            actor_salvavidas = random.randint(0, num_actores - 1)
            toma[actor_salvavidas] = 1

        matriz_aleatoria.append(toma)

    return matriz_aleatoria

# PRUEBA DEL GENERADOR
# Vamos a inventarnos una "Súper Producción": 50 tomas y 15 actores
NUEVAS_TOMAS = 50
NUEVOS_ACTORES = 15

# Usamos la probabilidad de 0.25 (25% de chance de que un actor salga en una escena)
matriz_inventada = generar_guion_aleatorio(NUEVAS_TOMAS, NUEVOS_ACTORES, prob_aparicion=0.25)

print("NUEVO JUEGO DE DATOS GENERADO")
print(f"Película simulada con {len(matriz_inventada)} tomas y {len(matriz_inventada[0])} actores.")
print("\nEjemplo de las primeras 3 tomas generadas al azar (1=Participa, 0=No):")
for i in range(3):
    print(f"Toma {i+1}: {matriz_inventada[i]}")

NUEVO JUEGO DE DATOS GENERADO
Película simulada con 50 tomas y 15 actores.

Ejemplo de las primeras 3 tomas generadas al azar (1=Participa, 0=No):
Toma 1: [0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0]
Toma 2: [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]
Toma 3: [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0]


Aplica el algoritmo al juego de datos generado

Respuesta

A continuación, procedo a ejecutar el algoritmo Voraz (Greedy) diseñado anteriormente, pero esta vez pasándole como entrada la nueva matriz aleatoria generada en el paso previo (nuestra "superproducción" de 50 tomas y 15 actores).

El objetivo es demostrar que el algoritmo es **escalable y generalizable**. No está "ajustado" (overfitted) para funcionar únicamente con los datos del problema original, sino que puede agrupar óptimamente cualquier configuración de tomas y actores manteniendo los límites de las restricciones (máximo 6 tomas por día) y minimizando la cantidad de actores únicos a pagar por jornada.

In [56]:
# APLICACIÓN DEL ALGORITMO A LOS DATOS ALEATORIOS

print("PLANIFICACIÓN DE LA PELÍCULA ALEATORIA")

# Tomamos el tiempo de nuevo, solo por curiosidad
inicio_random = time.time()

# Aplicamos el algoritmo a los datos inventados
plan_random = algoritmo_voraz_doblaje(matriz_inventada)

# Calculamos el costo de esta nueva planificación
costo_random = costo_total_planificacion(plan_random, matriz_inventada)

fin_random = time.time()
tiempo_random = (fin_random - inicio_random) * 1000

print(f"Total de tomas procesadas: {len(matriz_inventada)}")
print(f"Total de días de grabación requeridos: {len(plan_random)}")
print(f"Costo total (Actores a pagar): {costo_random}")
print(f"Tiempo de ejecución: {tiempo_random:.4f} milisegundos\n")

print("Planificación detallada (Índices de las tomas por día):")
for i, dia in enumerate(plan_random):
    print(f"Día {i+1}: Tomas {dia}")

PLANIFICACIÓN DE LA PELÍCULA ALEATORIA
Total de tomas procesadas: 50
Total de días de grabación requeridos: 9
Costo total (Actores a pagar): 93
Tiempo de ejecución: 0.2370 milisegundos

Planificación detallada (Índices de las tomas por día):
Día 1: Tomas [4, 38, 3, 7, 46, 0]
Día 2: Tomas [11, 12, 15, 19, 23, 29]
Día 3: Tomas [34, 42, 43, 8, 22, 26]
Día 4: Tomas [30, 39, 45, 48, 2, 13]
Día 5: Tomas [16, 18, 20, 21, 24, 25]
Día 6: Tomas [27, 28, 31, 32, 33, 35]
Día 7: Tomas [37, 41, 49, 1, 5, 6]
Día 8: Tomas [9, 10, 17, 36, 40, 47]
Día 9: Tomas [14, 44]


Enumera las referencias que has utilizado(si ha sido necesario) para llevar a cabo el trabajo

Respuesta

Para la resolución de este problema, el diseño de los algoritmos y el análisis de complejidad, me he apoyado en la siguiente bibliografía y documentación:

1. **Material de la asignatura:**
   * Universidad Internacional de Valencia (VIU). *03MIAR - Algoritmos de optimización: Trabajo Práctico*. Material de estudio y descripción de los problemas.

2. **Libros de referencia en Algoritmia (Bibliografía recomendada del curso):**
   * Brassard, G., & Bratley, P. (1997). *Fundamentos de Algoritmia*. Prentice Hall. (Consultado para la teoría de algoritmos Voraces y Vuelta Atrás).
   * Lee, R. C. T., Tseng, S. S., Chang, R. C., & Tsai, Y. T. (2005). *Introducción al diseño y análisis de algoritmos*.
   * Pérez Aguila, R. (2012). *Una introducción a las matemáticas para el análisis y diseño de algoritmos*. (Consultado para el cálculo de la complejidad polinómica y exponencial).

3. **Documentación técnica:**
   * Python Software Foundation. (2024). *Python Language Reference, version 3.x*. Recuperado de https://docs.python.org/3/ (Para el manejo de estructuras de datos `set`, librerías `random` y `time`).
   * Pandas Development Team. (2024). *Pandas Documentation*. Recuperado de https://pandas.pydata.org/ (Para la carga, lectura y filtrado del archivo CSV).

Describe brevemente las lineas de como crees que es posible avanzar en el estudio del problema. Ten en cuenta incluso posibles variaciones del problema y/o variaciones al alza del tamaño

Respuesta

**Respuesta:**

Para avanzar en el estudio de este problema y acercarlo a un entorno de producción cinematográfica real, propongo las siguientes líneas de investigación y desarrollo:

**1. Aplicación de Metaheurísticas:**
Aunque nuestro algoritmo Voraz es extremadamente rápido, tiene la limitación de que se conforma con el "óptimo local". Para avanzar, la línea ideal sería implementar algoritmos metaheurísticos como **Algoritmos Genéticos** o **Recocido Simulado**. Podríamos inyectar la solución rápida del algoritmo Voraz como "individuo inicial" de un Algoritmo Genético, permitiendo que la mutación y el cruce exploren combinaciones menos obvias que reduzcan aún más el costo final.

**2. Variaciones realistas del problema:**
* **Tarifas diferenciadas:** El problema actual asume que todos cobran igual. Una variación clave sería asignar un costo específico a cada actor (el protagonista cobra mucho más que un extra). La función objetivo ya no contaría "cantidad de actores", sino la "suma de los salarios".
* **Restricciones de agenda:** En la vida real, los actores tienen otros compromisos. Se podría añadir una restricción donde ciertos actores solo estén disponibles en días específicos de la semana.
* **Duración temporal (El problema de la Mochila):** Limitar a "6 tomas por día" es poco realista si una toma dura 5 minutos y otra 3 horas. Una evolución lógica sería asignar una duración en minutos a cada toma y establecer un límite de, por ejemplo, "480 minutos (8 horas) por jornada".

**3. Variaciones al alza del tamaño (Escalabilidad):**
Si escalamos el problema a una serie de televisión completa (ej. 10,000 tomas y 300 actores), la matriz binaria sería gigantesca. Aunque el algoritmo Voraz aguantaría por su complejidad polinómica, el margen de error respecto al óptimo global podría traducirse en miles de dólares de sobrecosto.
Para mitigarlo, se podría investigar un **Enfoque Híbrido con Machine Learning**: usar técnicas de Clustering (como K-Means o agrupamiento jerárquico) para pre-agrupar las tomas que comparten la mayor cantidad de actores, y luego aplicar el algoritmo Voraz únicamente dentro de cada clúster para hacer la asignación final de los días.